# Loan Approval Prediction using Python

## Import libraries

In [69]:
!pip install flask-ngrok

In [70]:
!pip install flask flask-ngrok pyngrok joblib

In [71]:
pip install flask scikit-learn joblib numpy pandas

In [72]:
from pyngrok import ngrok

ngrok.set_auth_token("3DvQUeKum8Yjg6v5IUyPoximHPE_3a8A6aTEmj1tudTaywzKB")

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Load Dataset

In [ ]:
df=pd.read_csv("/content/loan_approval.csv")

In [ ]:
df.head()

In [ ]:
df=df.drop("Applicant_ID", axis=1)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.describe())

In [ ]:
print(df.info())

## Handle Missing Values

In [ ]:
categorical_col=df.select_dtypes(include=['object']).columns
numerical_col=df.select_dtypes(include=['int64','float64']).columns

In [ ]:
categorical_col

In [ ]:
numerical_col

In [ ]:
from sklearn.impute import SimpleImputer
num_imputer=SimpleImputer(strategy='median')
df[numerical_col]=num_imputer.fit_transform(df[numerical_col])

In [ ]:
from sklearn.impute import SimpleImputer
cat_imputer=SimpleImputer(strategy='most_frequent')
df[categorical_col]=cat_imputer.fit_transform(df[categorical_col])

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

In [ ]:
print(df['Loan_Approved'].value_counts())

In [ ]:
import plotly.express as px

loan_status_count=df['Loan_Approved'].value_counts().reset_index()
loan_status_count.columns=["Loan_approved",'count']

# plot the pie chart
fig_loan_status=px.pie(loan_status_count, names='Loan_approved', values='count', title='Loan Status Distribution')

fig_loan_status.show()

In [ ]:
gender_count=df['Gender'].value_counts()
fig_gender=px.bar(gender_count,x=gender_count.index, y=gender_count.values,title='Gender distribution')

fig_gender.show()

In [ ]:
marital_count=df['Marital_Status'].value_counts()
fig_gender=px.bar(marital_count,x=marital_count.index, y=marital_count.values,title='Marital status distribution')

fig_gender.show()

In [ ]:
fig_applicant_income=px.histogram(df,x='Applicant_Income',title='Applicant Income Distribution')

fig_applicant_income.show()

In [ ]:
fig_coapplicant_income=px.histogram(df,x='Coapplicant_Income',title='Coapplicant Income Distribution')

fig_coapplicant_income.show()

In [ ]:
fig_income=px.box(df,x='Loan_Approved',y='Applicant_Income', color="Loan_Approved",title='Loan Approved vs Applicant Income')

fig_income.show()

In [ ]:
fig_credit_score=px.histogram(df, x='Credit_Score', color='Loan_Approved',barmode='group',title='Loan Approved vs Credit_score')
fig_credit_score.show()

## Encoding variables

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
label_encode=LabelEncoder()
for col in ["Marital_Status","Gender","Loan_Approved"]:
  if col in df.columns:
    df[col]=label_encode.fit_transform(df[col])

In [ ]:
df.head()

In [ ]:
df=pd.get_dummies(df,columns=[
    "Employment_Status",
    "Property_Area",
    "Education_Level",
    "Loan_Purpose",
    "Employer_Category"
],drop_first=True)

In [ ]:
df.head()

## Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_df=df.select_dtypes(include=['float64','int64'])
corr=numeric_df.corr()["Loan_Approved"].sort_values(ascending=False)
print(corr)

plt.figure(figsize=(10,5))
sns.heatmap(numeric_df.corr(),annot=True,cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

## Training Loan Approval Prediction Model

In [ ]:
df

In [ ]:
X=df.drop("Loan_Approved",axis=1)
y=df["Loan_Approved"]

In [ ]:
X

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.4,random_state=42)

### Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

### Training Models

In [ ]:
log_model=LogisticRegression(max_iter=1000,random_state=42)
log_model.fit(X_train_scaled,y_train)
log_pred=log_model.predict(X_test_scaled)

In [ ]:
svm_model=SVC(kernel="rbf",probability=True,random_state=42)
svm_model.fit(X_train_scaled,y_train)
svm_pred=svm_model.predict(X_test_scaled)

In [ ]:
rf_model=RandomForestClassifier(n_estimators=200,random_state=42)
rf_model.fit(X_train_scaled,y_train)
rf_pred=rf_model.predict(X_test_scaled)

### Evaluates Models

In [ ]:
models={
    "Logistic Regression":(y_test,log_pred),
    "Support Vector Machine":(y_test,svm_pred),
    "Random Forest":(y_test,rf_pred)
}

accuracy_scores={}

In [ ]:
for model_name, (y_true,y_pred) in models.items():
  acc=accuracy_score(y_true,y_pred)
  accuracy_scores[model_name]=acc
  print(f"{model_name} Accuracy: {acc}")
  print("Confusion Matrix:")
  print(confusion_matrix(y_true,y_pred))
  print("Classification Report:")
  print(classification_report(y_true,y_pred))
  print("\n")

### Best Model Selection

In [ ]:
best_model=max(accuracy_scores,key=accuracy_scores.get)
print(f"Best Model: {best_model}")

### Important Features

In [ ]:
importances=rf_model.feature_importances_

In [ ]:
importances

In [ ]:
importance_df=pd.DataFrame({
    "Feature":X.columns,
    "Importance":importances
}).sort_values(by="Importance",ascending=False)

In [ ]:
importance_df.head()

In [ ]:
import joblib

# Save model
joblib.dump(rf_model, 'loan_model.pkl')

# Save scaler
joblib.dump(X_train_scaled, 'scaler.pkl')

print("Model and scaler saved successfully")

In [ ]:
import joblib

joblib.dump(rf_model, 'loan_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

In [76]:
import joblib
model = joblib.load('loan_model.pkl')
# The model was trained with X_train_scaled (a numpy array), so it does not have feature_names_in_ directly.
# The original feature names are available in X.columns.
print("أسماء الأعمدة:", list(X.columns))

أسماء الأعمدة: ['Applicant_Income', 'Coapplicant_Income', 'Age', 'Marital_Status', 'Dependents', 'Credit_Score', 'Existing_Loans', 'DTI_Ratio', 'Savings', 'Collateral_Value', 'Loan_Amount', 'Loan_Term', 'Gender', 'Employment_Status_Salaried', 'Employment_Status_Self-employed', 'Employment_Status_Unemployed', 'Property_Area_Semiurban', 'Property_Area_Urban', 'Education_Level_Not Graduate', 'Loan_Purpose_Car', 'Loan_Purpose_Education', 'Loan_Purpose_Home', 'Loan_Purpose_Personal', 'Employer_Category_Government', 'Employer_Category_MNC', 'Employer_Category_Private', 'Employer_Category_Unemployed']


In [77]:
import pandas as pd

# Define all 27 feature values for a single sample
# User provided: Applicant_Income, Coapplicant_Income, Loan_Amount, Loan_Term, Credit_Score (interpreted as 1)
sample_values = {
    'Applicant_Income': 5000.0,
    'Coapplicant_Income': 2000.0,
    'Age': 40.0,  # Median from df.describe()
    'Marital_Status': 0, # Assuming 'Married' (0) based on previous encoding
    'Dependents': 1.0,  # Median from df.describe()
    'Credit_Score': 1.0, # User provided value (Note: This is very low for a typical credit score range 550-799)
    'Existing_Loans': 2.0, # Median from df.describe()
    'DTI_Ratio': 0.34,  # Median from df.describe()
    'Savings': 9880.5,  # Median from df.describe()
    'Collateral_Value': 24321.0, # Median from df.describe()
    'Loan_Amount': 120.0,
    'Loan_Term': 360.0,
    'Gender': 1, # Assuming 'Male' (1) based on previous encoding
    'Employment_Status_Private': 0, # Default to 0, if not specified
    'Employment_Status_Self-employed': 0, # Default to 0, if not specified
    'Employment_Status_Unemployed': 0, # Default to 0, if not specified
    'Property_Area_Semiurban': 0, # Default to 0, if not specified
    'Property_Area_Urban': 0, # Default to 0, if not specified
    'Education_Level_Not Graduate': 0, # Default to 0, if not specified
    'Loan_Purpose_Car': 0, # Default to 0, if not specified
    'Loan_Purpose_Education': 0, # Default to 0, if not specified
    'Loan_Purpose_Home': 0, # Default to 0, if not specified
    'Loan_Purpose_Personal': 0, # Default to 0, if not specified
    'Employer_Category_Government': 0, # Default to 0, if not specified
    'Employer_Category_MNC': 0, # Default to 0, if not specified
    'Employer_Category_Private': 0, # Default to 0, if not specified
    'Employer_Category_Unemployed': 0 # Default to 0, if not specified
}

# Create a DataFrame from the sample values, ensuring the correct column order
sample_df = pd.DataFrame([sample_values], columns=X.columns)

sample_scaled = scaler.transform(sample_df)

prediction = rf_model.predict(sample_scaled)

if prediction[0] == 1:
    print("Loan Approved")
else:
    print("Loan Rejected")

Loan Rejected
